In [ ]:
# =========================
# HELPER
# =========================

def safe_int(x):
    if pd.isna(x):
        return 0
    try:
        return int(float(x))
    except:
        return 0

def pad_str(x):
    return str(x).zfill(3)

In [ ]:
# =========================
# BUILD DATA SLS
# =========================

hasil = []
logic_all = []
operan_all = []

for kode_kab in sheet_kabkota:
    print(f"\nMemproses Kab/Kota {kode_kab}")
    df_sls = df_sls_all[kode_kab].copy()
    df_logic_kab = df_logic[
        df_logic["kdkab"] == kode_kab
    ].copy()
    logic_all.append(df_logic_kab)

    for col in ["kdkec","kddesa"]:
        df_sls[col] = (
            df_sls[col]
            .astype(str)
            .str.zfill(3)
        )

        df_logic_kab[col] = (
            df_logic_kab[col]
            .astype(str)
            .str.zfill(3)
        )

    # =========================
    # PARSE OPERAN
    # =========================

    for _, row in df_logic_kab.iterrows():
        ket = str(row.get("Keterangan Operan",""))

        # DEBUG — tampilkan isi mentah
        if "+" in ket:
            print("ISI KETERANGAN:", repr(ket))

        matches = re.findall(
            r'\+\s*(\d+)\s*SLS\s*dari\s*desa\s*(\d{3})-(\d{3})',
            ket
        )

        for m in matches:
            operan_all.append({
                "kdkec_asal": m[1],
                "kddesa_asal": m[2],
                "kdkec_tujuan": row["kdkec"],
                "kddesa_tujuan": row["kddesa"],
                "jumlah": int(m[0])
            })

            print(
                "Kab", kode_kab,
                "operan terbaca:",
                len(operan_all)
            )

    # =========================
    # BUILD UNIT SLS
    # =========================

    for _, r in df_sls.iterrows():
        hasil.append({
            "_row_order": r.get("_row_order",0),
            "kdkab": kode_kab,
            "kdkec": pad_str(r["kdkec"]),
            "kddesa": pad_str(r["kddesa"]),
            "kdkec_petugas": pad_str(r["kdkec"]),
            "kddesa_petugas": pad_str(r["kddesa"]),
            "kdsls": r["kdsls"],
            "kdsubsls": r["kdsubsls"],
            "kk": safe_int(r["kk"]),
            "usaha_rev": safe_int(r["usaha_rev"]),
            "status_petugas": "Reguler"
        })

df_hasil = pd.DataFrame(hasil)

df_logic_all = pd.concat(
    logic_all,
    ignore_index=True
)

df_operan_map = pd.DataFrame(
    operan_all
)

# =========================
# AUTOMATIC SPECIAL CASE
# CANCEL OPERAN IF DON'T NECESSARY
# =========================

print("\nValidasi kebutuhan operan")

desa_capacity = {}

for _, row in df_logic_all.iterrows():
    key = (
        pad_str(row["kdkec"]),
        pad_str(row["kddesa"])
    )

    petugas = safe_int(
        row["Petugas revisi"]
    )

    kapasitas = petugas * 20

    desa_capacity[key] = kapasitas

desa_active_sls = defaultdict(int)

for _, r in df_hasil.iterrows():
    if (r["kk"] > 0) or (r["usaha_rev"] > 0):
        key = (
            r["kdkec"],
            r["kddesa"]
        )
        desa_active_sls[key] += 1

In [ ]:
# =========================
# USE EXACTLY THE SAME OPERAN AS IN LOGIKADASAR
# =========================

operan_valid = operan_all.copy()

print(
    "Kab",
    kode_kab,
    "operan valid:",
    len(operan_valid)
)

df_operan_map = pd.DataFrame(
    operan_valid
)

In [ ]:
# =========================
# OPERAN NORMAL (WITH CAPACITY VALIDATION)
# =========================

print("\nMemproses Operan")

df_hasil["_tersedia"] = True

df_operan_map = pd.DataFrame(
    operan_valid,
    columns=[
        "kdkec_asal",
        "kddesa_asal",
        "kdkec_tujuan",
        "kddesa_tujuan",
        "jumlah"
    ]
)

if df_operan_map.empty:
    print("Tidak ada operan")

else:

    group_oper = df_operan_map.groupby(
        ["kdkec_asal","kddesa_asal"]
    )

    for (kec_asal, desa_asal), df_grp in group_oper:

        kec_asal = pad_str(kec_asal)
        desa_asal = pad_str(desa_asal)

        key_asal = (
            kec_asal,
            desa_asal
        )

        # =========================
        # CALCULATE THE ACTIVE SLS OF ORIGINAL VILLAGE
        # =========================

        mask_asal_all = (
            (df_hasil["kdkec"] == kec_asal) &
            (df_hasil["kddesa"] == desa_asal)
        )

        df_asal_all = df_hasil[
            mask_asal_all
        ]

        df_asal_aktif = df_asal_all[
            (df_asal_all["kk"] > 0) |
            (df_asal_all["usaha_rev"] > 0)
        ]

        jumlah_sls_aktif = len(
            df_asal_aktif
        )

        # =========================
        # ORIGINAL VILLAGE CAPACITY
        # =========================

        kapasitas_asal = desa_capacity.get(
            key_asal,
            0
        )

        # =========================
        # MAXIMUM NUMBER ALLOWED TO BE MOVED
        # =========================

        max_boleh_oper = max(
            0,
            jumlah_sls_aktif
            - kapasitas_asal
        )

        if max_boleh_oper == 0:
            continue

        # =========================
        # PUT THE AVAILABLE SLS
        # =========================

        mask_tersedia = (
            mask_asal_all &
            (df_hasil["_tersedia"])
        )

        df_asal = df_hasil[
            mask_tersedia
        ]

        df_asal = df_asal.sort_values(
            ["kk","usaha_rev"],
            ascending=False
        )

        pointer = 0

        total_dipindah = 0

        for _, op in df_grp.iterrows():

            jumlah_diminta = safe_int(
                op["jumlah"]
            )

            # LIMIT SO IT DOES NOT EXCEED THE MAXIMUM NUMBER
            jumlah = min(
                jumlah_diminta,
                max_boleh_oper
                - total_dipindah
            )

            if jumlah <= 0:
                break

            tujuan_kec = pad_str(
                op["kdkec_tujuan"]
            )

            tujuan_desa = pad_str(
                op["kddesa_tujuan"]
            )

            pointer_end = min(
                pointer + jumlah,
                len(df_asal)
            )

            df_pindah = df_asal.iloc[
                pointer:pointer_end
            ]

            for i in df_pindah.index:

                df_hasil.loc[
                    i,
                    "kdkec_petugas"
                ] = tujuan_kec

                df_hasil.loc[
                    i,
                    "kddesa_petugas"
                ] = tujuan_desa

                df_hasil.loc[
                    i,
                    "status_petugas"
                ] = "Operan"

                df_hasil.loc[
                    i,
                    "_tersedia"
                ] = False

            pointer += jumlah
            total_dipindah += jumlah

In [ ]:
# =========================
# REBUILD MAPPING
# =========================

mapping = defaultdict(list)

for i,r in df_hasil.iterrows():
    key = (
        r["kdkec_petugas"],
        r["kddesa_petugas"]
    )
    mapping[key].append(i)

In [ ]:
# =========================
# ASSIGN PETUGAS
# =========================

df_hasil["no_urut_petugas"] = None

for _, desa in tqdm(
        df_logic_all.iterrows(),
        total=len(df_logic_all),
        ncols=100
):
    key = (
        pad_str(desa["kdkec"]),
        pad_str(desa["kddesa"])
    )
    idx_list = mapping.get(key,[])
    if not idx_list:
        continue

    beban_dasar = safe_int(
        desa[
            "Beban Dasar SLS Per Petugas (hasil)"
        ]
    )
    sisa = safe_int(
        desa[
            "Sisa SLS (hasil)"
        ]
    )

    if beban_dasar == 0:
        continue

    n_petugas = safe_int(
        desa["Petugas revisi"]
    )
    # TARGET
    target_list = []

    for i in range(n_petugas):
        if i < sisa:
            target_list.append(
                beban_dasar + 1
            )
        else:
            target_list.append(
                beban_dasar
            )

    unit = df_hasil.loc[idx_list]

    unit_valid = unit[
        (unit["kk"] > 0) |
        (unit["usaha_rev"] > 0)
    ].copy()

    if unit_valid.empty:
      continue

    unit_valid["total"] = (
        unit_valid["kk"] +
        unit_valid["usaha_rev"]
    )

    unit_valid = unit_valid.sort_values(
        "total",
        ascending=False
    )

    beban_sls = {p:0 for p in range(1,n_petugas+1)}
    beban_kk = {p:0 for p in range(1,n_petugas+1)}
    beban_usaha = {p:0 for p in range(1,n_petugas+1)}

    for idx,r in unit_valid.iterrows():
        kandidat = [
            p for p in range(1,n_petugas+1)
            if beban_sls[p]
               < target_list[p-1]
        ]

        if not kandidat:
            continue

        if n_petugas > 1:
            def score(p):
              total_petugas = (
                  beban_kk[p] +
                  beban_usaha[p]
              )

              total_sls = (
                  r["kk"] +
                  r["usaha_rev"]
              )

              return (
                  total_petugas +
                  total_sls
              )

            p_best = min(
                kandidat,
                key=score
            )

        else:
            p_best = kandidat[0]

        df_hasil.loc[
            idx,
            "no_urut_petugas"
        ] = p_best

        beban_sls[p_best] += 1
        beban_kk[p_best] += r["kk"]
        beban_usaha[p_best] += r["usaha_rev"]

# =========================
# ACTIVE FAILSAFE WITHOUT OFFICIAL
# =========================

mask_missing = (
    df_hasil["no_urut_petugas"].isna()
    &
    (
        (df_hasil["kk"] > 0)
        |
        (df_hasil["usaha_rev"] > 0)
    )
)

if mask_missing.sum() > 0:
    print(
        "PERINGATAN:",
        mask_missing.sum(),
        "SLS aktif belum dapat petugas"
    )

    df_hasil.loc[
        mask_missing,
        "no_urut_petugas"
    ] = 1

# =========================
# FAILSAFE NON ACTIVE SLS
# =========================

df_hasil.loc[
    (df_hasil["kk"] == 0) &
    (df_hasil["usaha_rev"] == 0),
    "no_urut_petugas"
] = None

In [ ]:
# =========================
# OFFICIAL CODE
# =========================

df_hasil["kode_petugas"] = (
    df_hasil["kdkab"].astype(str)
    + df_hasil["kdkec_petugas"]
    + df_hasil["kddesa_petugas"]
    + df_hasil["no_urut_petugas"]
        .apply(
            lambda x:
            str(int(x)).zfill(2)
            if pd.notna(x)
            else ""
        )
)

# =========================
# OPERAN DESCRIPTION COLUMN
# =========================

def build_keterangan(r):
    # If different villages → Operan
    if (
        r["kdkec"] != r["kdkec_petugas"]
        or
        r["kddesa"] != r["kddesa_petugas"]
    ):
        return (
            f"Operan dari "
            f"{r['kdkec']}-{r['kddesa']} | "
            f"Petugas desa "
            f"{r['kdkec_petugas']}-{r['kddesa_petugas']}"
        )
    return ""

# =========================
# OFFICER DESCRIPTION
# =========================

def build_keterangan_petugas(r):
    if r["status_petugas"] == "Operan":
        if pd.notna(r["kode_petugas"]):
            return (
                f"Dikerjakan oleh petugas "
                f"{r['kode_petugas']}"
            )
    return ""


df_hasil["keterangan"] = df_hasil.apply(
    build_keterangan,
    axis=1
)

df_hasil["keterangan_petugas"] = df_hasil.apply(
    build_keterangan_petugas,
    axis=1
)

In [ ]:
# =========================
# FINAL OFFICER STATUS
# (make sure this remains)
# =========================

df_hasil["status_petugas"] = df_hasil["status_petugas"].fillna("Reguler")

# =========================
# FINAL DATA READY FOR EXPORT
# =========================

df_final = df_hasil[[
    "keterangan",
    "kdkab",
    "kdkec",
    "kddesa",
    "kdkec_petugas",
    "kddesa_petugas",
    "kdsls",
    "kdsubsls",
    "kk",
    "usaha_rev",
    "no_urut_petugas",
    "kode_petugas",
    "keterangan_petugas",
    "status_petugas"
]]


print("\nSELESAI ✅")

#Validation

In [ ]:
# =========================
# FILTER SLS WITH OFFICERS
# =========================
df_rekap_petugas = df_hasil[
    df_hasil["no_urut_petugas"].notna()
].copy()

# =========================
# SUMMARY BY OFFICER
# =========================
df_rekap_petugas = (
    df_rekap_petugas
    .groupby([
        "kdkab",
        "kdkec",
        "kddesa_petugas",
        "no_urut_petugas",
        "kode_petugas"
    ])
    .agg(
        jumlah_sls = ("kdsls", "count"),
        total_kk = ("kk", "sum"),
        total_usaha = ("usaha_rev", "sum")
    )
    .reset_index()
    .sort_values(["kdkec","kddesa_petugas","no_urut_petugas"])
)

# =========================
# SHOW THE RESULTS
# =========================
print("\nREKAP BEBAN PETUGAS:")
print(df_rekap_petugas.head(20))

#Visualization of Data Distribution Balance

##Distribution of Households vs. Tasks per Officer

The balance between the number of households and tasks assigned to each officer forms a linear upward trend, indicating greater balance.

In [ ]:
df_plot = df_rekap_petugas.copy()

plt.figure()
plt.scatter(df_plot["total_kk"], df_plot["total_usaha"])

plt.xlabel("Total KK")
plt.ylabel("Total Usaha")
plt.title("Distribusi KK vs Usaha per Petugas")

judul = f"{kode_kab_uji}_kk vs usaha"

filepath = os.path.join(
    image,
    f"{judul}.png"
)

plt.savefig(
    filepath,
    dpi=300,
    bbox_inches="tight"
)

print("Plot disimpan ke:", filepath)

plt.show()
plt.close()

##Distribution of the Number of SLS Per Officer

distribution of the SLS workload for each officer

In [ ]:
plt.figure()
plt.hist(df_plot["jumlah_sls"])

plt.xlabel("Jumlah SLS")
plt.ylabel("Frekuensi")
plt.title("Distribusi Jumlah SLS per Petugas")

judul = f"{kode_kab_uji}_dist sls"

filepath = os.path.join(
    image,
    f"{judul}.png"
)

plt.savefig(
    filepath,
    dpi=300,
    bbox_inches="tight"
)

print("Plot disimpan ke:", filepath)

plt.show()
plt.close()

##Check whether there are still any officer members with a workload exceeding 20 SLS

In [ ]:
# =========================
# FILTER OFFICER WITH > 20 SLS
# =========================
df_over_sls = df_rekap_petugas[
    df_rekap_petugas["jumlah_sls"] > 20
].copy()

# =========================
# SORT
# =========================
df_over_sls = df_over_sls.sort_values(
    "jumlah_sls", ascending=False
)

# =========================
# SHOW THE RESULT
# =========================
print("\nPETUGAS DENGAN SLS > 20:")
print(df_over_sls)

print("\nJumlah petugas overload:", len(df_over_sls))

##Distribution of Deviation in Number of Households from Target and Distribution of Deviation in Number of Businesses from Target

- the greater the number of officers whose number of households is close to the target, the closer the value is to 0, indicating that officers are approaching the target for number of households
- the workload of officers for the number of businesses close to the target; the closer to 0, the closer the officers’ performance is to the target for the number of businesses

In [ ]:
# =========================
# MERGE TARGETS FROM LOGIC
# =========================
df_plot = df_rekap_petugas.merge(
    df_logic[[
        "kdkab","kdkec","kddesa",
        "target_kk_per_petugas",
        "target_usaha_per_petugas"
    ]],
    left_on=["kdkab","kdkec","kddesa_petugas"],
    right_on=["kdkab","kdkec","kddesa"],
    how="left"
)

# =========================
# ENSURE NUMERIC FORMAT (AFTER MERGING!)
# =========================
cols_numeric = [
    "total_kk",
    "total_usaha",
    "target_kk_per_petugas",
    "target_usaha_per_petugas"
]

for col in cols_numeric:
    df_plot[col] = (
        df_plot[col]
        .astype(str)
        .str.replace(",", "")
        .str.strip()
    )
    df_plot[col] = pd.to_numeric(df_plot[col], errors="coerce").fillna(0)

# =========================
# CALCULATE THE DIFFERENCE
# =========================
df_plot["selisih_kk"] = (
    df_plot["total_kk"] -
    df_plot["target_kk_per_petugas"]
)

df_plot["selisih_usaha"] = (
    df_plot["total_usaha"] -
    df_plot["target_usaha_per_petugas"]
)

In [ ]:
# a. histogram of the difference in the number of families
import matplotlib.pyplot as plt

plt.figure()
plt.hist(df_plot["selisih_kk"])

plt.xlabel("Selisih KK (Realisasi - Target)")
plt.ylabel("Frekuensi")
plt.title("Distribusi Deviasi KK terhadap Target")

plt.axvline(0)                                                                    # target line
plt.grid()

judul = f"{kode_kab_uji}_deviasi kk target"

filepath = os.path.join(
    image,
    f"{judul}.png"
)

plt.savefig(
    filepath,
    dpi=300,
    bbox_inches="tight"
)

print("Plot disimpan ke:", filepath)

plt.show()
plt.close()


# b. histogram of the difference in the number of businesses
plt.figure()
plt.hist(df_plot["selisih_usaha"])

plt.xlabel("Selisih Usaha (Realisasi - Target)")
plt.ylabel("Frekuensi")
plt.title("Distribusi Deviasi Usaha terhadap Target")

plt.axvline(0)                                                                    # target line
plt.grid()

judul = f"{kode_kab_uji}_deviasi usaha target"

filepath = os.path.join(
    image,
    f"{judul}.png"
)

plt.savefig(
    filepath,
    dpi=300,
    bbox_inches="tight"
)

print("Plot disimpan ke:", filepath)

plt.show()
plt.close()


# histogram centered at 0 -> target and actual are close

##Deviation of Number of Families vs. Number of Businesses per Officer

difference between the number of businesses and the number of families relative to the target; the more concentrated the distribution, the closer both are to the target

In [ ]:
# scatter: deviation between the number of families and the number of businesses
plt.figure()

plt.scatter(
    df_plot["selisih_kk"],
    df_plot["selisih_usaha"]
)

plt.xlabel("Selisih KK")
plt.ylabel("Selisih Usaha")
plt.title("Deviasi KK vs Usaha per Petugas")

plt.axhline(0)                                                                    # horizontal target line
plt.axvline(0)                                                                    # vertical target line

plt.grid()

judul = f"{kode_kab_uji}_deviasi kk usaha petugas"

filepath = os.path.join(
    image,
    f"{judul}.png"
)

plt.savefig(
    filepath,
    dpi=300,
    bbox_inches="tight"
)

print("Plot disimpan ke:", filepath)

plt.show()
plt.close()


#  scatter clusters in the center -> ideal distribution

#Save The Results to Excel

In [ ]:
# =========================
# SORT DATA (SAFE)
# =========================
if "_row_order" in df_hasil.columns:
    df_hasil = df_hasil.sort_values("_row_order").reset_index(drop=True)
else:
    df_hasil = df_hasil.reset_index(drop=True)

# =========================
# SUMMARY OF OPERAN
# =========================
df_operan = df_hasil[df_hasil["status_petugas"] == "Operan"].copy()

if not df_operan.empty:

    # =========================
    # SELECT HOME VILLAGE (SAFE)
    # =========================
    extract_df = df_operan["keterangan"].str.extract(
        r'Operan(?: petugas)? dari kec (\d{3}) desa (\d{3})'
    )

    df_operan["desa_asal"] = extract_df.apply(
        lambda x: f"{x[0]}-{x[1]}" if pd.notna(x[0]) and pd.notna(x[1]) else None,
        axis=1
    )

    # =========================
    # SUMMARY
    # =========================
    df_rekap = (
        df_operan
        .groupby([
            "kdkab",
            "kdkec",
            "kddesa_petugas",
            "no_urut_petugas",
            "kode_petugas",
            "desa_asal"
        ], dropna=False)
        .size()
        .reset_index(name="jumlah_sls_operan")
        .sort_values(["kdkec","kddesa_petugas","no_urut_petugas"])
        .reset_index(drop=True)
    )

else:
    df_rekap = pd.DataFrame(columns=[
        "kdkab","kdkec","kddesa_petugas",
        "no_urut_petugas","kode_petugas",
        "desa_asal","jumlah_sls_operan"
    ])

# =========================
# EXPORT
# =========================
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_file = os.path.join(
    OUTPUT_DIR,
    f"hasil_pembagian_sls_final_{kode_kab_uji}.xlsx"
)

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:

    # =========================
    # SHEET 1: DETAIL
    # =========================
    df_hasil.drop(columns=["_row_order"], errors="ignore").to_excel(
        writer,
        sheet_name="hasil_detail",
        index=False
    )

    # =========================
    # SHEET 2: SUMMARY OF OPERAN
    # =========================
    df_rekap.to_excel(
        writer,
        sheet_name="rekap_operan",
        index=False
    )

print("\n✅ File berhasil disimpan di:")
print(output_file)